In [1]:
!pip install -q -U transformers datasets accelerate bitsandbytes tqdm peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 21.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, math, random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm

from peft import PeftModel

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Torch:", torch.__version__)


Device: cuda
Torch: 2.9.0+cu126


In [4]:
PHASE2_DRIVE_PATH = "/content/drive/My Drive/ML_Project_Phase2/phi3_squad_final"
base_model_id = "microsoft/Phi-3.5-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(PHASE2_DRIVE_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, PHASE2_DRIVE_PATH)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print("Loaded base:", base_model_id)
print("Loaded Phase2 adapters from:", PHASE2_DRIVE_PATH)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Loaded base: microsoft/Phi-3.5-mini-instruct
Loaded Phase2 adapters from: /content/drive/My Drive/ML_Project_Phase2/phi3_squad_final


In [5]:
class EarlyExitException(Exception):
    def __init__(self, hidden_state):
        super().__init__("Early exit triggered")
        self.hidden_state = hidden_state

def unwrap_peft(m):
    if hasattr(m, "base_model"):
        m = m.base_model
    if hasattr(m, "model"):
        m = m.model
    return m

def get_transformer_blocks(model):
    candidates = [model, unwrap_peft(model)]
    for m in candidates:
        if hasattr(m, "model") and hasattr(m.model, "layers"):
            return m.model.layers
        if hasattr(m, "layers"):
            return m.layers
        if hasattr(m, "transformer") and hasattr(m.transformer, "h"):
            return m.transformer.h
        if hasattr(m, "transformer") and hasattr(m.transformer, "layers"):
            return m.transformer.layers
    raise ValueError("Could not locate transformer blocks. Print unwrap_peft(model) to inspect.")

def forward_until_block(model, input_ids, attention_mask=None, block_index=8):
    blocks = get_transformer_blocks(model)
    assert 0 <= block_index < len(blocks), f"block_index {block_index} out of range (0..{len(blocks)-1})"

    def hook_fn(module, inputs, output):
        hs = output[0] if isinstance(output, (tuple, list)) else output
        raise EarlyExitException(hs)

    handle = blocks[block_index].register_forward_hook(hook_fn)
    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=False,
        )
    except EarlyExitException as e:
        return e.hidden_state
    finally:
        handle.remove()

    raise RuntimeError("EarlyExitException was not triggered — hook did not run as expected.")


In [6]:
class EarlyExitHead(nn.Module):
    def __init__(self, hidden_size: int, vocab_size: int, dropout: float = 0.1):
        super().__init__()
        self.ln = nn.LayerNorm(hidden_size)
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, vocab_size),
        )

    def forward(self, hidden_states):
        x = self.ln(hidden_states)
        return self.ff(x)

hidden_size = getattr(model.config, "hidden_size", None) or getattr(model.config, "n_embd", None)
vocab_size = model.config.vocab_size
print("hidden_size:", hidden_size, "vocab_size:", vocab_size)

head = EarlyExitHead(hidden_size, vocab_size).to(device)  # ✅ fp32 head
print("✅ head dtype:", next(head.parameters()).dtype)


hidden_size: 3072 vocab_size: 32064
✅ head dtype: torch.float32


In [7]:
raw = load_dataset("squad", split="train[:4000]")

def format_squad(example):
    ans = example["answers"]["text"][0] if len(example["answers"]["text"]) > 0 else ""
    text = (
        "<|user|>\n"
        f"Context: {example['context']}\n"
        f"Question: {example['question']}\n"
        "<|assistant|>\n"
        f"{ans}<|end|>"
    )
    return {"text": text}

raw = raw.map(format_squad, remove_columns=raw.column_names)
split = raw.train_test_split(test_size=0.05, seed=SEED)
train_ds, val_ds = split["train"], split["test"]

MAX_LEN = 256
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN, padding=False)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_ds   = val_ds.map(tokenize, batched=True, remove_columns=["text"])

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

print("Train size:", len(train_ds), "Val size:", len(val_ds))


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Train size: 3800 Val size: 200


In [8]:
def collate_fn(features):
    batch = tokenizer.pad(features, padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=2, shuffle=False, collate_fn=collate_fn)

loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

def early_exit_logits(input_ids, attention_mask):
    # Run model forward under autocast (fast), but head in fp32
    with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
        hs = forward_until_block(model, input_ids=input_ids, attention_mask=attention_mask, block_index=8)

        if hasattr(model, "model") and hasattr(model.model, "norm"):
            try:
                hs = model.model.norm(hs)
            except Exception:
                pass

    # ✅ Cast to FP32 for the head (fixes dtype issues + avoids GradScaler FP16 unscale errors)
    hs = hs.float()

    # Ensure head runs in full precision
    with torch.amp.autocast("cuda", enabled=False):
        return head(hs)

@torch.no_grad()
def evaluate():
    head.eval()
    total_loss, total_tokens = 0.0, 0

    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = early_exit_logits(input_ids, attention_mask)

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        loss = loss_fn(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )

        ntok = (shift_labels != -100).sum().item()
        total_loss += loss.item() * max(ntok, 1)
        total_tokens += max(ntok, 1)

    ppl = math.exp(total_loss / total_tokens) if total_tokens > 0 else float("inf")
    return (total_loss / total_tokens), ppl


In [9]:
optimizer = torch.optim.AdamW(head.parameters(), lr=5e-4, weight_decay=0.01)
print("✅ Optimizer params:", sum(p.numel() for g in optimizer.param_groups for p in g["params"]))


✅ Optimizer params: 107979072


In [10]:
EPOCHS = 1
GRAD_ACCUM = 8
MAX_STEPS = 237

head.train()
optimizer.zero_grad(set_to_none=True)

global_step = 0
progress = tqdm(total=min(MAX_STEPS, EPOCHS * len(train_loader)), desc="Training")

for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = early_exit_logits(input_ids, attention_mask)

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        loss = loss_fn(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )
        loss = loss / GRAD_ACCUM

        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            progress.update(1)
            progress.set_postfix({"loss": float(loss.detach().cpu())})

        if global_step >= MAX_STEPS:
            break

    if global_step >= MAX_STEPS:
        break

progress.close()

val_loss, val_ppl = evaluate()
print(f"Val loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}")


Training:   0%|          | 0/237 [00:00<?, ?it/s]

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Val loss: 1.1405 | Val PPL: 3.13


In [11]:
from google.colab import drive
import os

PHASE3_DRIVE_DIR = "/content/drive/My Drive/ML_Project_Phase3"
os.makedirs(PHASE3_DRIVE_DIR, exist_ok=True)

print("Saving Phase 3 artifacts to:", PHASE3_DRIVE_DIR)

save_path = os.path.join(PHASE3_DRIVE_DIR, "early_exit_head_block9.pt")

torch.save({
    "base_model_id": base_model_id,
    "phase2_adapter_path": PHASE2_DRIVE_PATH,
    "block_index": 8,  # 9th transformer block
    "hidden_size": hidden_size,
    "vocab_size": vocab_size,
    "max_len": MAX_LEN,
    "head_state_dict": head.state_dict(),
    "head_dtype": str(next(head.parameters()).dtype),
}, save_path)

print("Saved early-exit head to Drive:", save_path)

Saving Phase 3 artifacts to: /content/drive/My Drive/ML_Project_Phase3
Saved early-exit head to Drive: /content/drive/My Drive/ML_Project_Phase3/early_exit_head_block9.pt


In [13]:
import pandas as pd

@torch.no_grad()
def next_token_with_early_exit(input_ids, attention_mask, threshold=0.6):
    head.eval()

    ee_logits = early_exit_logits(input_ids, attention_mask)
    last_logits = ee_logits[:, -1, :]

    probs = torch.softmax(last_logits, dim=-1)
    conf, pred = probs.max(dim=-1)

    if conf.item() >= threshold:
        return pred, True, conf.item()

    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    full_last = out.logits[:, -1, :]
    pred_full = torch.argmax(full_last, dim=-1)
    return pred_full, False, conf.item()

@torch.no_grad()
def generate_with_stats(prompt, max_new_tokens=40, threshold=0.6):
    model.eval()
    head.eval()

    enc = tokenizer(prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    # List to store details for every generated token
    token_details = []
    early_exit_count = 0

    for _ in range(max_new_tokens):
        # 1. Get prediction and exit decision
        nxt, exited, conf = next_token_with_early_exit(input_ids, attention_mask, threshold=threshold)

        # 2. Decode the specific token we just generated
        token_str = tokenizer.decode(nxt, skip_special_tokens=False)

        # 3. Store the stats
        token_details.append({
            "Token": token_str,
            "Exit Layer": "Early (L9)" if exited else "Full (L32)",
            "Confidence": f"{conf:.2%}" # Format as percentage
        })

        if exited:
          early_exit_count+=1


        # 4. Update inputs for next step
        input_ids = torch.cat([input_ids, nxt.view(1, 1)], dim=1)
        attention_mask = torch.cat(
            [attention_mask, torch.ones((1,1), device=device, dtype=attention_mask.dtype)], dim=1
        )

        if nxt.item() == tokenizer.eos_token_id:
            break

    full_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return full_text, token_details, early_exit_count

# --- Run the Demo ---
test_context = "The Apollo program was a human spaceflight program carried out by NASA. It succeeded in landing the first humans on the Moon from 1969 to 1972."
test_question = "When did the Apollo program land humans on the Moon?"
prompt = f"<|user|>\nContext: {test_context}\n\nQuestion: {test_question}<|end|>\n<|assistant|>\n"
print(f"Prompt:\n{prompt}")

# Run generation
out_text, stats, early_exit_count = generate_with_stats(prompt, max_new_tokens=30, threshold=0.8) # Higher threshold = harder to exit early

print(f"\nModel Answer:\n{out_text}")
print("\nEarly-exit used at tokens:", early_exit_count, "/", len(stats))
print("\n--- Per-Token Exit Decisions ---\n")

# Display as a Pandas DataFrame (Looks like a table)
df = pd.DataFrame(stats)
print(df.to_markdown(index=False))

Prompt:
<|user|>
Context: The Apollo program was a human spaceflight program carried out by NASA. It succeeded in landing the first humans on the Moon from 1969 to 1972.

Question: When did the Apollo program land humans on the Moon?<|end|>
<|assistant|>


Model Answer:
Context: The Apollo program was a human spaceflight program carried out by NASA. It succeeded in landing the first humans on the Moon from 1969 to 1972.

Question: When did the Apollo program land humans on the Moon? 1969 to 1972

Early-exit used at tokens: 4 / 13

--- Per-Token Exit Decisions ---

| Token         | Exit Layer   | Confidence   |
|:--------------|:-------------|:-------------|
|               | Full (L32)   | 7.96%        |
| 1             | Full (L32)   | 36.39%       |
| 9             | Full (L32)   | 20.55%       |
| 6             | Full (L32)   | 27.48%       |
| 9             | Full (L32)   | 25.05%       |
| to            | Full (L32)   | 63.93%       |
|               | Early (L9)   | 91.97%      